# Product Review Rating Prediction
## Fine-Tuning RoBERTa vs. BiGRU Baseline — Amazon Reviews (5-class)

**Single notebook — run top to bottom.** No carry-forward needed.

### What this notebook does (in order):
1. Loads `SetFit/amazon_reviews_multi_en` (200K train / 5K val / 5K test)
2. Classic NLP preprocessing → vocabulary + fastText embedding matrix (for BiGRU)
3. RoBERTa tokenization (256-token and 128-token splits)
4. Trains BiGRU + Additive Attention baseline (2 seeds)
5. Fine-tunes RoBERTa-base with mean pooling (2 seeds)
6. Ablation study: Mean vs. CLS pooling; 128 vs. 256 max length
7. Full evaluation: Accuracy, Macro F1, MAE, confusion matrices, qualitative analysis, token importance visualization

---
## Part 0 — Install & Imports

In [ ]:
!pip install -q datasets transformers[torch] accelerate spacy scikit-learn seaborn
!python -m spacy download en_core_web_sm -q

In [ ]:
import os, re, pickle, random, collections, json, time, zipfile
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import (
    RobertaModel, RobertaTokenizerFast, DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from tqdm.auto import tqdm

# ── Global settings ────────────────────────────────────────────────────────────
SEEDS       = [42, 123]
NUM_CLASSES = 5
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP     = torch.cuda.is_available()

# RoBERTa compute budget (full data = 2445 s/epoch; 60K subset = ~730 s/epoch).
# Set RB_TRAIN_N = None to use all 200K reviews.
RB_TRAIN_N = 60000
RB_EPOCHS  = 2

os.makedirs('outputs', exist_ok=True)

print(f'Device  : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'AMP     : {USE_AMP}')
print(f'PyTorch : {torch.__version__}')

---
## Part 1 — Load Dataset
> `SetFit/amazon_reviews_multi_en` — English Amazon reviews, 5-class star rating

In [ ]:
print('Loading SetFit/amazon_reviews_multi_en ...')
dataset  = load_dataset('SetFit/amazon_reviews_multi_en')
train_ds = dataset['train']       # 200,000 reviews
val_ds   = dataset['validation']  #   5,000 reviews
test_ds  = dataset['test']        #   5,000 reviews

print(dataset)
print(f'\nTrain: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}')
print('Sample:', train_ds[0])
print('Columns:', train_ds.column_names)
assert 'text' in train_ds.column_names, 'unexpected schema'
assert len(train_ds[0]['text'].strip()) > 0, 'empty text - check column names'

# Label distribution
counts = collections.Counter(train_ds['label'])
print('\nLabel distribution (train):', dict(sorted(counts.items())))
plt.figure(figsize=(6, 3))
plt.bar([f'{k+1}★' for k in sorted(counts)], [counts[k] for k in sorted(counts)], color='steelblue')
plt.title('Training Label Distribution'); plt.xlabel('Star Rating'); plt.ylabel('Count')
plt.tight_layout(); plt.savefig('outputs/label_dist.png', dpi=150); plt.show()

---
## Part 2 — Classic NLP Preprocessing (for BiGRU Baseline)
Pipeline: clean → tokenize (spaCy) → build vocab → encode → load fastText embeddings

In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

HTML_RE  = re.compile(r'<[^>]+>')
URL_RE   = re.compile(r'https?://\S+|www\.\S+')
SPACE_RE = re.compile(r'\s+')

def clean_text(text: str) -> str:
    text = HTML_RE.sub(' ', text)
    text = URL_RE.sub(' ', text)
    return SPACE_RE.sub(' ', text.lower()).strip()

def tokenize_spacy(text: str):
    return [tok.text for tok in nlp(text) if not tok.is_space]

def get_text(sample) -> str:
    """Concatenate review_title + review_body as required by the task."""
    # SetFit/amazon_reviews_multi_en stores title + body already merged in 'text'
    if sample.get('text'):
        return sample['text'].strip()
    return ((sample.get('review_title') or '') + ' ' + (sample.get('review_body') or '')).strip()

# Quick smoke-test
ex  = train_ds[0]
raw = get_text(ex)
print('Raw    :', raw[:150])
print('Clean  :', clean_text(raw)[:150])
print('Tokens :', tokenize_spacy(clean_text(raw))[:15])

In [ ]:
MAX_SEQ_LEN = 200
MIN_FREQ    = 3
PAD_TOKEN   = '<PAD>'
UNK_TOKEN   = '<UNK>'

print('Building vocabulary from training set ...')
word_freq       = collections.Counter()
train_tok_cache = []   # cache tokenized training texts to avoid re-tokenizing
train_labels_cl = []   # labels for classic pipeline

for sample in tqdm(train_ds, desc='Tokenising train'):
    toks = tokenize_spacy(clean_text(get_text(sample)))
    word_freq.update(toks)
    train_tok_cache.append(toks)
    train_labels_cl.append(sample['label'])

vocab    = [PAD_TOKEN, UNK_TOKEN] + [w for w, c in word_freq.items() if c >= MIN_FREQ]
word2idx = {w: i for i, w in enumerate(vocab)}
print(f'Vocabulary size: {len(vocab):,}')

In [ ]:
def encode_tokens(toks, max_len=MAX_SEQ_LEN):
    unk = word2idx[UNK_TOKEN]
    ids = [word2idx.get(t, unk) for t in toks[:max_len]]
    if len(ids) == 0: ids = [unk]  # prevent all-PAD row
    ids += [0] * (max_len - len(ids))   # 0 = PAD index
    return ids

# Encode all splits
train_ids_cl = np.array(
    [encode_tokens(t) for t in tqdm(train_tok_cache, desc='Encoding train')], dtype=np.int32
)
train_lbl_cl = np.array(train_labels_cl, dtype=np.int64)

def encode_split(split_ds):
    ids, lbls = [], []
    for s in tqdm(split_ds):
        ids.append(encode_tokens(tokenize_spacy(clean_text(get_text(s)))))
        lbls.append(s['label'])
    return np.array(ids, dtype=np.int32), np.array(lbls, dtype=np.int64)

val_ids_cl,  val_lbl_cl  = encode_split(val_ds)
test_ids_cl, test_lbl_cl = encode_split(test_ds)
print(f'train_ids: {train_ids_cl.shape}, val_ids: {val_ids_cl.shape}, test_ids: {test_ids_cl.shape}')

In [ ]:
# ── FastText Pre-trained Word Vectors ──────────────────────────────────────────
# These are SEPARATE from the review dataset.
# They are 300-dim word embeddings used to initialise the BiGRU embedding layer.
# (Task spec: 'pre-trained fastText or GloVe word embeddings as its input layer')
#
# Source: Meta AI, Common Crawl, 2 million English words, 300 dimensions.
# URL: https://fasttext.cc/docs/en/english-vectors.html

EMBED_DIM        = 300
FASTTEXT_VEC_URL = 'https://dl.fbaipublicfiles.com/fasttext/vectors-english/crawl-300d-2M.vec.zip'
FASTTEXT_LOCAL   = 'crawl-300d-2M.vec'
KAGGLE_FT_PATH   = '/kaggle/input/fasttext-crawl-300d-2m/crawl-300d-2M.vec'

# Priority 1: Kaggle dataset input attached to this notebook (no download)
# Priority 2: Auto-download from Meta AI (~1.6 GB zip → ~6 GB .vec)
import glob
# Any attached GloVe 840B (300d) or fastText dataset is picked up automatically
_found = (glob.glob('/kaggle/input/**/glove.840B.300d.txt', recursive=True) +
          glob.glob('/kaggle/input/**/crawl-300d-2M.vec', recursive=True))
if _found:
    embed_path = _found[0]
    print(f'Using Kaggle dataset input: {embed_path}')
elif os.path.exists(KAGGLE_FT_PATH):
    embed_path = KAGGLE_FT_PATH
    print(f'Using Kaggle dataset input: {embed_path}')
elif os.path.exists(FASTTEXT_LOCAL):
    embed_path = FASTTEXT_LOCAL
    print(f'Using cached local file: {embed_path}')
else:
    print('Downloading fastText vectors from Meta AI (~1.6 GB) ...')
    urllib.request.urlretrieve(FASTTEXT_VEC_URL, 'crawl-300d-2M.vec.zip')
    with zipfile.ZipFile('crawl-300d-2M.vec.zip', 'r') as z:
        z.extractall('.')
    os.remove('crawl-300d-2M.vec.zip')
    embed_path = FASTTEXT_LOCAL
    print(f'Done. File: {embed_path}')

print(f'Building {len(vocab):,} × {EMBED_DIM} embedding matrix ...')
embedding_matrix = np.random.normal(0, 0.1, (len(vocab), EMBED_DIM)).astype(np.float32)
embedding_matrix[0] = 0.0   # PAD → zero vector
hit = 0
with open(embed_path, encoding='utf-8', errors='ignore') as fh:
    for line in tqdm(fh, desc='Loading fastText vectors'):
        parts = line.rstrip().split(' ')
        if len(parts) != EMBED_DIM + 1:
            continue   # skip header row if present
        word = parts[0]
        if word in word2idx:
            embedding_matrix[word2idx[word]] = np.array(parts[1:], dtype=np.float32)
            hit += 1
print(f'Coverage: {hit}/{len(vocab)} = {hit/len(vocab)*100:.1f}%')
print(f'Embedding matrix shape: {embedding_matrix.shape}')

---
## Part 3 — RoBERTa Tokenization

In [ ]:
ROBERTA_MODEL = 'roberta-base'
tokenizer     = RobertaTokenizerFast.from_pretrained(ROBERTA_MODEL)
collator      = DataCollatorWithPadding(tokenizer, return_tensors='pt')

def make_tokenize_fn(max_len):
    def fn(examples):
        texts = [t or '' for t in examples['text']]
        return tokenizer(texts, max_length=max_len, truncation=True, padding=False)
    return fn

def tokenize_split(ds, max_len, desc=''):
    keep = [c for c in ds.column_names if c == 'label']
    drop = [c for c in ds.column_names if c != 'label']
    out  = ds.map(make_tokenize_fn(max_len), batched=True, batch_size=1000,
                  remove_columns=drop, desc=desc)
    out.set_format('torch')
    return out

# Same fixed subset for the main run AND the ablations, so they are directly comparable.
rb_src = train_ds if RB_TRAIN_N is None else train_ds.shuffle(seed=0).select(range(RB_TRAIN_N))
print(f'RoBERTa training reviews: {len(rb_src):,}')

print('Tokenising for RoBERTa (max_len=256) ...')
rb_train_256 = tokenize_split(rb_src, 256, 'train-256')
rb_val_256   = tokenize_split(val_ds,   256, 'val-256')
rb_test_256  = tokenize_split(test_ds,  256, 'test-256')

print('Tokenising for RoBERTa (max_len=128) — ablation ...')
rb_train_128 = tokenize_split(rb_src, 128, 'train-128')
rb_val_128   = tokenize_split(val_ds,   128, 'val-128')
rb_test_128  = tokenize_split(test_ds,  128, 'test-128')

print('Done. Columns:', rb_train_256.column_names)

---
## Part 4 — BiGRU + Additive Attention Baseline

In [ ]:
# ── Dataset ────────────────────────────────────────────────────────────────────
class ReviewDataset(Dataset):
    def __init__(self, ids, labels):
        self.ids    = torch.tensor(ids,    dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):  return len(self.ids)
    def __getitem__(self, i): return self.ids[i], self.labels[i]

def make_classic_loaders(batch_size=128):
    kw = dict(num_workers=2, pin_memory=True)
    return (
        DataLoader(ReviewDataset(train_ids_cl, train_lbl_cl), batch_size, shuffle=True,  **kw),
        DataLoader(ReviewDataset(val_ids_cl,   val_lbl_cl),   batch_size, shuffle=False, **kw),
        DataLoader(ReviewDataset(test_ids_cl,  test_lbl_cl),  batch_size, shuffle=False, **kw),
    )


# ── Model ──────────────────────────────────────────────────────────────────────
class AdditiveAttention(nn.Module):
    """Bahdanau-style additive attention pooling."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.w = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, h, pad_mask=None):
        # h: (B, T, H),  pad_mask: (B, T) True=PAD
        scores = self.v(torch.tanh(self.w(h))).squeeze(-1)   # (B, T)
        if pad_mask is not None:
            all_pad = pad_mask.all(dim=-1, keepdim=True)
            safe_mask = pad_mask & (~all_pad)
            scores = scores.masked_fill(safe_mask, -1e4)
        weights = scores.softmax(dim=-1)                      # (B, T)
        context = torch.bmm(weights.unsqueeze(1), h).squeeze(1)  # (B, H)
        return context, weights


class BiGRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim=256, num_layers=2,
                 num_classes=5, dropout=0.3, emb_weights=None, freeze=True):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        if emb_weights is not None:
            self.embedding.weight.data.copy_(torch.tensor(emb_weights, dtype=torch.float32))
        self.embedding.weight.requires_grad_(not freeze)

        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=num_layers,
                          batch_first=True, bidirectional=True,
                          dropout=dropout if num_layers > 1 else 0.0)
        self.attn       = AdditiveAttention(hidden_dim * 2)
        self.drop       = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, ids):
        pad_mask = (ids == 0)                           # (B, T)
        x = self.drop(self.embedding(ids))             # (B, T, E)
        h, _ = self.gru(x)                             # (B, T, 2H)
        ctx, _ = self.attn(h, pad_mask)                # (B, 2H)
        return self.classifier(self.drop(ctx))         # (B, 5)

    def unfreeze_embeddings(self):
        self.embedding.weight.requires_grad_(True)


# Sanity check
_m = BiGRUClassifier(len(vocab), EMBED_DIM, emb_weights=embedding_matrix)
print('BiGRU output shape:', _m(torch.zeros(2, MAX_SEQ_LEN, dtype=torch.long)).shape)
del _m

In [ ]:
# ── Evaluation helper ──────────────────────────────────────────────────────────
def eval_classic(model, loader):
    model.eval()
    preds, trues, total_loss = [], [], 0.0
    crit = nn.CrossEntropyLoss()
    with torch.no_grad():
        for ids, labels in loader:
            ids, labels = ids.to(DEVICE), labels.to(DEVICE)
            logits       = model(ids)
            total_loss  += crit(logits, labels).item() * len(labels)
            preds.extend(logits.argmax(-1).cpu().numpy())
            trues.extend(labels.cpu().numpy())
    preds, trues = np.array(preds), np.array(trues)
    return (total_loss / len(trues),
            accuracy_score(trues, preds),
            f1_score(trues, preds, average='macro', zero_division=0),
            float(np.mean(np.abs(preds - trues))),
            preds, trues)


# ── Training function ──────────────────────────────────────────────────────────
def train_bigru(seed, hidden_dim=256, lr=1e-3, batch_size=128,
                max_epochs=15, patience=3, grad_clip=1.0):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

    train_loader, val_loader, test_loader = make_classic_loaders(batch_size)
    model = BiGRUClassifier(len(vocab), EMBED_DIM, hidden_dim=hidden_dim,
                             emb_weights=embedding_matrix, freeze=True).to(DEVICE)
    optim    = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    sched    = torch.optim.lr_scheduler.ReduceLROnPlateau(optim, 'max', patience=1, factor=0.5)
    crit     = nn.CrossEntropyLoss()
    best_mf1, best_state, no_imp = -1.0, None, 0
    history  = dict(train_loss=[], val_loss=[], val_acc=[], val_mf1=[])

    for epoch in range(1, max_epochs + 1):
        if epoch == 2:                                  # unfreeze embeddings after epoch 1
            model.unfreeze_embeddings()
            optim.add_param_group({'params': model.embedding.parameters(), 'lr': lr * 0.1})
            print('  [embeddings unfrozen]')

        model.train(); ep_loss = 0.0; t0 = time.time()
        for ids, labels in train_loader:
            ids, labels = ids.to(DEVICE), labels.to(DEVICE)
            optim.zero_grad()
            loss = crit(model(ids), labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optim.step()
            ep_loss += loss.item() * len(labels)

        tr_loss = ep_loss / len(train_loader.dataset)
        vl, va, vmf1, vmae, _, _ = eval_classic(model, val_loader)
        sched.step(vmf1)
        for k, v in zip(['train_loss','val_loss','val_acc','val_mf1'], [tr_loss, vl, va, vmf1]):
            history[k].append(v)
        print(f'[BiGRU s{seed}] Ep{epoch:02d} [{time.time()-t0:.0f}s] '
              f'TrLoss={tr_loss:.4f} ValLoss={vl:.4f} ValAcc={va:.4f} ValMF1={vmf1:.4f} ValMAE={vmae:.4f}')

        if vmf1 > best_mf1:
            best_mf1  = vmf1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_imp    = 0
            torch.save(best_state, f'outputs/bigru_best_seed{seed}.pt')
        else:
            no_imp += 1
            if no_imp >= patience:
                print(f'  Early stopping at epoch {epoch}'); break

    model.load_state_dict(best_state)
    _, acc, mf1, mae, preds, trues = eval_classic(model, test_loader)
    print(f'[BiGRU s{seed}] TEST Acc={acc:.4f} MF1={mf1:.4f} MAE={mae:.4f}')
    return history, acc, mf1, mae, preds, trues

In [ ]:
bigru_results = {}
for seed in SEEDS:
    cache = f'outputs/bigru_res_s{seed}.pkl'
    if os.path.exists(cache):                       # finished in an earlier run
        bigru_results[seed] = pickle.load(open(cache, 'rb')); print(f'BiGRU seed {seed}: loaded cache'); continue
    print(f'\n{"="*65}\nBiGRU SEED {seed}\n{"="*65}')
    h, acc, mf1, mae, preds, trues = train_bigru(seed)
    bigru_results[seed] = dict(history=h, acc=acc, mf1=mf1, mae=mae, preds=preds, trues=trues)
    pickle.dump(bigru_results[seed], open(cache, 'wb'))

bg_accs = [bigru_results[s]['acc'] for s in SEEDS]
bg_mf1s = [bigru_results[s]['mf1'] for s in SEEDS]
bg_maes = [bigru_results[s]['mae'] for s in SEEDS]
print(f'\n=== BiGRU Final (mean ± std over {len(SEEDS)} seeds) ===')
print(f'Accuracy : {np.mean(bg_accs):.4f} ± {np.std(bg_accs):.4f}')
print(f'Macro F1 : {np.mean(bg_mf1s):.4f} ± {np.std(bg_mf1s):.4f}')
print(f'MAE      : {np.mean(bg_maes):.4f} ± {np.std(bg_maes):.4f}')

In [ ]:
# Training curves — BiGRU
best_bg_seed = max(SEEDS, key=lambda s: bigru_results[s]['mf1'])
h = bigru_results[best_bg_seed]['history']
ep = range(1, len(h['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(ep, h['train_loss'], 'o-', label='Train Loss')
axes[0].plot(ep, h['val_loss'],   's-', label='Val Loss')
axes[0].set(xlabel='Epoch', ylabel='Loss', title='BiGRU — Loss'); axes[0].legend(); axes[0].grid(True)
axes[1].plot(ep, h['val_acc'], 'o-', label='Val Accuracy')
axes[1].plot(ep, h['val_mf1'], 's-', label='Val Macro F1')
axes[1].set(xlabel='Epoch', ylabel='Score', title='BiGRU — Val Metrics'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.savefig('outputs/bigru_training_curves.png', dpi=150); plt.show()

---
## Part 5 — RoBERTa Fine-Tuning

In [ ]:
# ── Model ──────────────────────────────────────────────────────────────────────
class RoBERTaClassifier(nn.Module):
    """
    RoBERTa-base with masked mean pooling over the last hidden layer.
    pooling='mean': masked mean of all non-PAD token embeddings (default, required by task)
    pooling='cls' : first-token <s> embedding (ablation)
    """
    def __init__(self, model_name='roberta-base', num_classes=5, dropout=0.1, pooling='mean'):
        super().__init__()
        assert pooling in ('mean', 'cls')
        self.roberta    = RobertaModel.from_pretrained(model_name)
        self.pooling    = pooling
        self.drop       = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.roberta.config.hidden_size, num_classes)

    def mean_pool(self, last_hidden, attention_mask):
        mask  = attention_mask.unsqueeze(-1).float()
        return (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

    def forward(self, input_ids, attention_mask):
        lh     = self.roberta(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        pooled = self.mean_pool(lh, attention_mask) if self.pooling == 'mean' else lh[:, 0]
        return self.classifier(self.drop(pooled))


# ── DataLoaders ────────────────────────────────────────────────────────────────
def make_rb_loaders(tr, va, te, batch_size=16):
    kw = dict(collate_fn=collator, num_workers=2, pin_memory=True)
    return (
        DataLoader(tr, batch_size, shuffle=True,  **kw),
        DataLoader(va, batch_size, shuffle=False, **kw),
        DataLoader(te, batch_size, shuffle=False, **kw),
    )


# ── Evaluation ─────────────────────────────────────────────────────────────────
def eval_roberta(model, loader):
    model.eval()
    preds, trues, total_loss = [], [], 0.0
    crit = nn.CrossEntropyLoss()
    with torch.no_grad():
        for batch in loader:
            lk     = 'labels' if 'labels' in batch else 'label'
            labels = batch.pop(lk).to(DEVICE)
            ids    = batch['input_ids'].to(DEVICE)
            mask   = batch['attention_mask'].to(DEVICE)
            logits = model(ids, mask)
            total_loss += crit(logits, labels).item() * len(labels)
            preds.extend(logits.argmax(-1).cpu().numpy())
            trues.extend(labels.cpu().numpy())
    preds, trues = np.array(preds), np.array(trues)
    return (total_loss / len(trues),
            accuracy_score(trues, preds),
            f1_score(trues, preds, average='macro', zero_division=0),
            float(np.mean(np.abs(preds - trues))),
            preds, trues)


# ── Training function ──────────────────────────────────────────────────────────
def train_roberta(seed, pooling='mean', max_len=256,
                  lr=2e-5, weight_decay=0.01, batch_size=16, grad_accum=2,
                  max_epochs=RB_EPOCHS, patience=2, warmup_ratio=0.06, dropout=0.1, tag=''):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

    tr, va, te = (rb_train_128, rb_val_128, rb_test_128) if max_len == 128 \
                 else (rb_train_256, rb_val_256, rb_test_256)
    train_ldr, val_ldr, test_ldr = make_rb_loaders(tr, va, te, batch_size)

    model = RoBERTaClassifier(ROBERTA_MODEL, NUM_CLASSES, dropout, pooling).to(DEVICE)

    no_decay    = ['bias', 'LayerNorm.weight']
    optim_grps  = [
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
         'weight_decay': weight_decay},
        {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
         'weight_decay': 0.0},
    ]
    optim       = torch.optim.AdamW(optim_grps, lr=lr)
    total_steps = (len(train_ldr) // grad_accum) * max_epochs
    sched       = get_linear_schedule_with_warmup(optim, int(total_steps * warmup_ratio), total_steps)
    crit        = nn.CrossEntropyLoss()
    scaler      = torch.cuda.amp.GradScaler() if USE_AMP else None
    label       = tag or f'{pooling}_len{max_len}_s{seed}'

    best_mf1, best_state, no_imp = -1.0, None, 0
    history = dict(train_loss=[], val_loss=[], val_acc=[], val_mf1=[])

    for epoch in range(1, max_epochs + 1):
        model.train(); ep_loss = 0.0; t0 = time.time(); optim.zero_grad()
        for step, batch in enumerate(train_ldr):
            lk     = 'labels' if 'labels' in batch else 'label'
            labels = batch.pop(lk).to(DEVICE)
            ids    = batch['input_ids'].to(DEVICE)
            mask   = batch['attention_mask'].to(DEVICE)

            if USE_AMP:
                with torch.cuda.amp.autocast():
                    loss = crit(model(ids, mask), labels) / grad_accum
                scaler.scale(loss).backward()
            else:
                loss = crit(model(ids, mask), labels) / grad_accum
                loss.backward()

            ep_loss += loss.item() * grad_accum * len(labels)
            if (step + 1) % grad_accum == 0:
                if USE_AMP:
                    scaler.unscale_(optim)
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optim); scaler.update()
                else:
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optim.step()
                sched.step(); optim.zero_grad()

        tr_loss = ep_loss / len(train_ldr.dataset)
        vl, va, vmf1, vmae, _, _ = eval_roberta(model, val_ldr)
        for k, v in zip(['train_loss','val_loss','val_acc','val_mf1'], [tr_loss, vl, va, vmf1]):
            history[k].append(v)
        print(f'[{label}] Ep{epoch:02d} [{time.time()-t0:.0f}s] '
              f'TrLoss={tr_loss:.4f} ValLoss={vl:.4f} ValAcc={va:.4f} ValMF1={vmf1:.4f} ValMAE={vmae:.4f}')

        if vmf1 > best_mf1:
            best_mf1  = vmf1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_imp    = 0
            torch.save(best_state, f'outputs/roberta_best_{label}.pt')
        else:
            no_imp += 1
            if no_imp >= patience:
                print(f'  Early stopping at epoch {epoch}'); break

    model.load_state_dict(best_state)
    _, acc, mf1, mae, preds, trues = eval_roberta(model, test_ldr)
    print(f'[{label}] TEST Acc={acc:.4f} MF1={mf1:.4f} MAE={mae:.4f}')
    return history, acc, mf1, mae, preds, trues, model

In [ ]:
roberta_results = {}
for seed in SEEDS:
    cache = f'outputs/roberta_res_s{seed}.pkl'
    if os.path.exists(cache):
        roberta_results[seed] = pickle.load(open(cache, 'rb')); print(f'RoBERTa seed {seed}: loaded cache'); continue
    print(f'\n{"="*65}\nRoBERTa (mean, 256) SEED {seed}\n{"="*65}')
    h, acc, mf1, mae, preds, trues, _m = train_roberta(
        seed, pooling='mean', max_len=256, tag=f'mean256_s{seed}'
    )
    del _m; torch.cuda.empty_cache()
    roberta_results[seed] = dict(history=h, acc=acc, mf1=mf1, mae=mae, preds=preds, trues=trues)
    pickle.dump(roberta_results[seed], open(cache, 'wb'))

# model used for token-importance: reload the best seed's saved checkpoint
_best = max(SEEDS, key=lambda s: roberta_results[s]['mf1'])
rb_model = RoBERTaClassifier(ROBERTA_MODEL, NUM_CLASSES, 0.1, 'mean').to(DEVICE)
rb_model.load_state_dict(torch.load(f'outputs/roberta_best_mean256_s{_best}.pt', map_location=DEVICE))
rb_model.eval()

rb_accs = [roberta_results[s]['acc'] for s in SEEDS]
rb_mf1s = [roberta_results[s]['mf1'] for s in SEEDS]
rb_maes = [roberta_results[s]['mae'] for s in SEEDS]
print(f'\n=== RoBERTa Main Results (mean pooling, 256 tokens) ===')
print(f'Accuracy : {np.mean(rb_accs):.4f} ± {np.std(rb_accs):.4f}')
print(f'Macro F1 : {np.mean(rb_mf1s):.4f} ± {np.std(rb_mf1s):.4f}')
print(f'MAE      : {np.mean(rb_maes):.4f} ± {np.std(rb_maes):.4f}')

In [ ]:
# Training curves — RoBERTa
best_rb_seed = max(SEEDS, key=lambda s: roberta_results[s]['mf1'])
h = roberta_results[best_rb_seed]['history']
ep = range(1, len(h['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(ep, h['train_loss'], 'o-', label='Train Loss')
axes[0].plot(ep, h['val_loss'],   's-', label='Val Loss')
axes[0].set(xlabel='Epoch', ylabel='Loss', title='RoBERTa — Loss'); axes[0].legend(); axes[0].grid(True)
axes[1].plot(ep, h['val_acc'], 'o-', label='Val Accuracy')
axes[1].plot(ep, h['val_mf1'], 's-', label='Val Macro F1')
axes[1].set(xlabel='Epoch', ylabel='Score', title='RoBERTa — Val Metrics'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.savefig('outputs/roberta_training_curves.png', dpi=150); plt.show()

---
## Part 6 — Ablation Study
### A) Mean Pooling vs. CLS Pooling
### B) Max Length 256 vs. 128

In [ ]:
def run_abl(tag, **kw):
    cache = f'outputs/abl_{tag}.pkl'
    if os.path.exists(cache): return pickle.load(open(cache, 'rb'))
    _, acc, mf1, mae, _, _, _m = train_roberta(42, tag=tag, **kw)
    del _m; torch.cuda.empty_cache()
    pickle.dump((acc, mf1, mae), open(cache, 'wb'))
    return acc, mf1, mae

print('=== Ablation A: CLS Pooling, max_len=256, seed=42 ===')
cls_acc, cls_mf1, cls_mae = run_abl('cls256_s42', pooling='cls', max_len=256)

print('\n=== Ablation B: Mean Pooling, max_len=128, seed=42 ===')
len128_acc, len128_mf1, len128_mae = run_abl('mean128_s42', pooling='mean', max_len=128)

mean256_acc = roberta_results[42]['acc']
mean256_mf1 = roberta_results[42]['mf1']
mean256_mae = roberta_results[42]['mae']

print(f'\n{"─"*60}')
print(f'{"Config":<30} {"Acc":>8} {"MF1":>8} {"MAE":>8}')
print(f'{"─"*60}')
print(f'{"Mean Pool, len=256 (main)":<30} {mean256_acc:>8.4f} {mean256_mf1:>8.4f} {mean256_mae:>8.4f}')
print(f'{"CLS Pool,  len=256 (ablation A)":<30} {cls_acc:>8.4f} {cls_mf1:>8.4f} {cls_mae:>8.4f}')
print(f'{"Mean Pool, len=128 (ablation B)":<30} {len128_acc:>8.4f} {len128_mf1:>8.4f} {len128_mae:>8.4f}')
print(f'{"─"*60}')

---
## Part 7 — Evaluation & Analysis

In [ ]:
# Pick best seed results
best_bg = bigru_results[max(SEEDS, key=lambda s: bigru_results[s]['mf1'])]
best_rb = roberta_results[best_rb_seed]
bg_preds, bg_trues = best_bg['preds'], best_bg['trues']
rb_preds, rb_trues = best_rb['preds'], best_rb['trues']

# ── Classification reports ─────────────────────────────────────────────────────
star_names = [f'{i+1}-star' for i in range(5)]
print('=== BiGRU Classification Report ===')
print(classification_report(bg_trues, bg_preds, target_names=star_names))
print('=== RoBERTa Classification Report ===')
print(classification_report(rb_trues, rb_preds, target_names=star_names))

In [ ]:
# ── Confusion matrices (side-by-side) ─────────────────────────────────────────
def norm_cm(preds, trues):
    cm = confusion_matrix(trues, preds)
    return cm.astype(float) / cm.sum(axis=1, keepdims=True)

labels = [f'{i+1}★' for i in range(5)]
bg_cm  = norm_cm(bg_preds, bg_trues)
rb_cm  = norm_cm(rb_preds, rb_trues)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, cm, title, cmap in zip(axes, [bg_cm, rb_cm],
                                ['BiGRU — Confusion Matrix', 'RoBERTa — Confusion Matrix'],
                                ['Oranges', 'Blues']):
    sns.heatmap(cm, annot=True, fmt='.2f', cmap=cmap, ax=ax,
                xticklabels=labels, yticklabels=labels)
    ax.set(xlabel='Predicted', ylabel='True', title=title)
plt.tight_layout()
plt.savefig('outputs/confusion_matrices.png', dpi=150)
plt.show()

In [ ]:
# ── Ordinal error analysis ─────────────────────────────────────────────────────
def ordinal_analysis(preds, trues, label):
    d = np.abs(np.array(preds) - np.array(trues))
    print(f'\nOrdinal Error Analysis — {label}')
    for name, cond in [('Correct   (diff=0)', d==0), ('Adjacent  (diff=1)', d==1),
                        ('Near-miss (diff=2)', d==2), ('Distant   (diff≥3)', d>=3)]:
        print(f'  {name}: {cond.mean():.3f}')

ordinal_analysis(bg_preds, bg_trues, 'BiGRU')
ordinal_analysis(rb_preds, rb_trues, 'RoBERTa')

In [ ]:
# ── Summary table ──────────────────────────────────────────────────────────────
print(f'\n{"═"*65}')
print(f'{"Model":<30} {"Accuracy":>12} {"Macro F1":>12} {"MAE":>8}')
print(f'{"─"*65}')
print(f'{"BiGRU (Baseline)":<30} '
      f'{np.mean(bg_accs):.4f}±{np.std(bg_accs):.4f}  '
      f'{np.mean(bg_mf1s):.4f}±{np.std(bg_mf1s):.4f}  '
      f'{np.mean(bg_maes):.4f}±{np.std(bg_maes):.4f}')
print(f'{"RoBERTa (Mean Pool, 256)":<30} '
      f'{np.mean(rb_accs):.4f}±{np.std(rb_accs):.4f}  '
      f'{np.mean(rb_mf1s):.4f}±{np.std(rb_mf1s):.4f}  '
      f'{np.mean(rb_maes):.4f}±{np.std(rb_maes):.4f}')
print(f'{"═"*65}')

---
## Part 8 — Qualitative Analysis

In [ ]:
# Align on test set indices
n = min(len(bg_preds), len(rb_preds), len(test_ds))
bg_p = np.array(bg_preds[:n]); rb_p = np.array(rb_preds[:n]); true = np.array(bg_trues[:n])

bf_rw = np.where((bg_p != true) & (rb_p == true))[0]   # BiGRU fails, RoBERTa wins
rf_bw = np.where((rb_p != true) & (bg_p == true))[0]   # RoBERTa fails, BiGRU wins
print(f'BiGRU fails / RoBERTa wins: {len(bf_rw):,}')
print(f'RoBERTa fails / BiGRU wins: {len(rf_bw):,}')

def show_examples(indices, title, n=5):
    rng = np.random.default_rng(0)
    chosen = rng.choice(indices, size=min(n, len(indices)), replace=False)
    print(f'\n{"═"*70}\n{title}\n{"═"*70}')
    for i, idx in enumerate(chosen):
        s = test_ds[int(idx)]
        text = get_text(s)
        print(f'\n[{i+1}] True={true[idx]+1}★  BiGRU={bg_p[idx]+1}★  RoBERTa={rb_p[idx]+1}★')
        print(f'Text: {text[:350]}')

show_examples(bf_rw, 'BiGRU FAILS — RoBERTa SUCCEEDS  (≥5 examples)', n=5)
show_examples(rf_bw, 'RoBERTa FAILS — BiGRU SUCCEEDS  (≥2 examples)', n=2)

---
## Part 9 — Token Importance Visualization (RoBERTa)

In [ ]:
rb_model.eval()

def token_importance(model, text, true_label):
    """Gradient-norm saliency w.r.t. input embeddings."""
    inp  = tokenizer(text, return_tensors='pt', max_length=256, truncation=True).to(DEVICE)
    emb  = model.roberta.embeddings(inp['input_ids'])
    emb.retain_grad()
    lh   = model.roberta(inputs_embeds=emb, attention_mask=inp['attention_mask']).last_hidden_state
    mask = inp['attention_mask'].unsqueeze(-1).float()
    p    = (lh * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
    logits = model.classifier(model.drop(p))
    logits[0, true_label].backward()

    imp    = emb.grad[0].detach().cpu().norm(dim=-1).numpy()
    tokens = tokenizer.convert_ids_to_tokens(inp['input_ids'][0].cpu().tolist())
    n      = len(tokens)
    imp    = (imp[:n] - imp[:n].min()) / (np.ptp(imp[:n]) + 1e-9)
    pred   = logits.argmax().item() + 1

    fig, ax = plt.subplots(figsize=(max(10, n // 3), 3))
    ax.bar(range(n), imp, color='steelblue')
    ax.set_xticks(range(n)); ax.set_xticklabels(tokens, rotation=90, fontsize=8)
    ax.set(ylabel='Gradient Importance',
           title=f'Token Importance  |  True: {true_label+1}★  Pred: {pred}★')
    plt.tight_layout()
    plt.savefig('outputs/token_importance.png', dpi=150)
    plt.show()


if len(bf_rw) > 0:
    idx = int(bf_rw[0])
    s   = test_ds[idx]
    txt = get_text(s)
    print(f'Visualising idx={idx} | True={true[idx]+1}★')
    print(f'Text: {txt[:200]}')
    token_importance(rb_model, txt, int(true[idx]))

---
## Part 10 — Save All Results

In [ ]:
results = {
    'bigru': {
        'acc_mean': float(np.mean(bg_accs)), 'acc_std': float(np.std(bg_accs)),
        'mf1_mean': float(np.mean(bg_mf1s)), 'mf1_std': float(np.std(bg_mf1s)),
        'mae_mean': float(np.mean(bg_maes)), 'mae_std': float(np.std(bg_maes)),
    },
    'roberta_main': {
        'acc_mean': float(np.mean(rb_accs)), 'acc_std': float(np.std(rb_accs)),
        'mf1_mean': float(np.mean(rb_mf1s)), 'mf1_std': float(np.std(rb_mf1s)),
        'mae_mean': float(np.mean(rb_maes)), 'mae_std': float(np.std(rb_maes)),
    },
    'ablation': {
        'mean_pool_256': {'acc': float(mean256_acc), 'mf1': float(mean256_mf1), 'mae': float(mean256_mae)},
        'cls_pool_256':  {'acc': float(cls_acc),     'mf1': float(cls_mf1),     'mae': float(cls_mae)},
        'mean_pool_128': {'acc': float(len128_acc),  'mf1': float(len128_mf1),  'mae': float(len128_mae)},
    }
}
with open('outputs/all_results.json', 'w') as f:
    json.dump(results, f, indent=2)

np.save('outputs/bigru_test_preds.npy',   bg_preds)
np.save('outputs/bigru_test_trues.npy',   bg_trues)
np.save('outputs/roberta_test_preds.npy', rb_preds)
np.save('outputs/roberta_test_trues.npy', rb_trues)

print('All results saved to outputs/')
print(json.dumps(results, indent=2))